# Submitted model — offline Kaggle inference

Attach the competition, a dataset with this repository snapshot, and a dataset with the release weights (submitted reads four files; `FINAL = "three-verifiers"` reads six). Enable a GPU and turn off internet access. Edit the three mounted paths in the first cell. If packages are missing, include a compatible wheelhouse under the code dataset and point `WHEELHOUSE` at it. The notebook writes `/kaggle/working/submission.csv`.

In [ ]:
import subprocess
import sys
from pathlib import Path

CODE = Path("/kaggle/input/REPLACE_CODE_DATASET/filament-segmentation-2026")
WEIGHTS = Path("/kaggle/input/REPLACE_WEIGHT_DATASET")
IMAGES = Path("/kaggle/input/REPLACE_COMPETITION_MOUNT/test_images")
WHEELHOUSE = None  # Set a Path to pre-downloaded wheels if needed.
OUTPUT = Path("/kaggle/working/submission.csv")
FINAL = "submitted"  # the final; "three-verifiers" replays the three-verifier variant

unets = [WEIGHTS / f"effv2s_s{seed}_fold-1.pt" for seed in range(2)]
verifiers = [WEIGHTS / f"verifier_resnet18_s{seed}.pt" for seed in range(1 if FINAL == "submitted" else 3)]
booster = WEIGHTS / "own_ver3_a05_l025_nometa_lgb.txt"
for path in [CODE / "submit.py", IMAGES, booster, *unets, *verifiers]:
    if not path.exists():
        raise FileNotFoundError(path)
if WHEELHOUSE is not None:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--no-index", "--find-links",
        str(WHEELHOUSE), "-r", str(CODE / "requirements.txt"),
    ], check=True)

In [ ]:
subprocess.run([
    sys.executable, str(CODE / "submit.py"), "--images", str(IMAGES),
    "--final", FINAL, "--weights", *map(str, unets), "--selector", str(booster),
    "--verifier-weights", *map(str, verifiers), "--out", str(OUTPUT),
], cwd=CODE, check=True)